<a href="https://colab.research.google.com/github/Koushikjains/Book-Ratings-Analytics-using-PySpark/blob/main/Bigdata_project_Team_28_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install dependencies
!pip install pyspark
!pip install -U -q PyDrive
!apt-get update -qq
!apt install openjdk-8-jdk-headless -qq

# 2. Set Java Environment
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# 3. Initialize Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .master("local[*]")\
    .appName("Amazon_Books_Analytics")\
    .config('spark.ui.port', '4050')\
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")\
    .getOrCreate()

print(f"Spark version: {spark.version}")

# 4. Setup Ngrok Tunnel with YOUR Token
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -o ngrok-stable-linux-amd64.zip
!pip install pyngrok
from pyngrok import ngrok

# --- YOUR SPECIFIC TOKEN ---
auth_token = "34XpsfNsgUR9vphCTy0rfdFA7Fd_25waUaU6rEY4wLn8GPgP2"
ngrok.set_auth_token(auth_token)

# Kill any existing tunnels to avoid errors
ngrok.kill()

# Start new tunnel
public_url = ngrok.connect(4050)
print("Spark UI is live at:", public_url)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 65 not upgraded.
Need to get 39.7 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Selecting previously unselected package libxtst6:amd64.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously uns

In [ ]:
# ====================================================
# STEP 0: CONNECT TO GOOGLE DRIVE
# ====================================================
from google.colab import drive
# This will ask for permission to access your Drive files
drive.mount('/content/drive')

# ====================================================
# STEP A: LOAD DATA (UPDATED PATHS FOR DRIVE)
# ====================================================
from pyspark.sql.functions import (col, count, when, isnan, regexp_replace,
                                   regexp_extract, split, size, mean, first,
                                   desc, lit, stddev, avg)
import datetime

print("--- Loading Data from Google Drive ---")

# PATHS UPDATED HERE: Pointing to /content/drive/MyDrive/...
books_df = spark.read.csv("/content/drive/MyDrive/books_data.csv", header=True, inferSchema=True, quote='"', escape='"')
ratings_df = spark.read.csv("/content/drive/MyDrive/Books_rating.csv", header=True, inferSchema=True, quote='"', escape='"')

# OPTIMIZATION: Cache books_df as it's used for Fill Rate AND Cleaning
books_df.cache()

# ====================================================
# STEP B: FILL RATE REPORT (Data Quality)
# ====================================================
print("\n--- Fill Rate Report ---")
total_rows = books_df.count()

def get_fill_rate(df):
    for c in df.columns:
        # Count missing values (Null, NaN, or Empty String)
        missing_count = df.filter(
            col(c).isNull() | isnan(c) | (col(c) == "")
        ).count()
        # Calculate Fill Rate
        fill_rate = (1 - (missing_count / total_rows)) * 100
        print(f"Column '{c}': {fill_rate:.2f}%")

get_fill_rate(books_df)

# ====================================================
# STEP C & D: DATA CLEANING & LOGIC CHECKS
# ====================================================
print("\n--- Cleaning Data ---")

# 1. Clean 'publishedDate': Extract just the Year (Regex)
books_clean = books_df.withColumn("year", regexp_extract(col("publishedDate"), r"(\d{4})", 1).cast("int"))

# 2. Clean 'authors': Remove brackets [' ']
books_clean = books_clean.withColumn("authors_clean", regexp_replace(col("authors"), r"\[|\]|'", "")) \
                         .withColumn("authors_array", split(col("authors_clean"), ","))

# 3. Logic Filter (Step D)
# Rule: Year must be valid (1800-2025) and Title cannot be Null
current_year = datetime.datetime.now().year
books_final = books_clean.filter(
    (col("Title").isNotNull()) &
    (col("year") > 1800) &
    (col("year") <= current_year)
)

# OPTIMIZATION: Cache the final clean data
books_final.cache()

# ====================================================
# STEP E: CLEANLINESS SCORE
# ====================================================
final_count = books_final.count()
cleanliness = (final_count / total_rows) * 100
print(f"\nData Cleanliness Score: {cleanliness:.2f}% (Rows retained: {final_count}/{total_rows})")

# ====================================================
# STEP F: INDUSTRY REQUIREMENT (Custom)
# ====================================================
# Requirement: "Identify Controversial Books (High Rating Volatility)"
print("\n--- Industry Req: Controversial Books (High Std Dev) ---")

volatility_df = ratings_df.groupBy("Title").agg(
    stddev("review/score").alias("volatility"),
    count("Title").alias("review_count")
)

controversial = volatility_df.filter(col("review_count") > 50) \
                             .orderBy(col("volatility").desc())

controversial.show(5, truncate=False)

# ====================================================
# SPECIFIC ASSIGNMENT REQUIREMENTS (Team 9/28)
# ====================================================

# --- Task A: Date Range & Sort ---
print("\n--- Task A Output: Books (1995-2005) Sorted by Rating ---")

start_year = 1995
end_year = 2005

task_a = books_final.filter(
    (col("year") >= start_year) &
    (col("year") <= end_year)
).select(
    col("Title"),
    col("ratingsCount").alias("avg_rating") # Metadata rating
).orderBy(col("avg_rating").desc())

task_a.show(5, truncate=False)


# --- Task B: Multi-Author Mean vs Avg ---
print("\n--- Task B Output: Multi-Author Analysis ---")

# 1. Filter Multi-Author Books
multi_author = books_final.filter(size(col("authors_array")) > 1) \
    .select(col("Title"), col("ratingsCount").alias("metadata_avg_rating"))

# 2. Calculate Real Mean from Reviews Table
calculated_stats = ratings_df.groupBy("Title").agg(
    mean("review/score").alias("calculated_mean_rating"),
    first("Id").alias("book_id")
)

# 3. Join
task_b = multi_author.join(calculated_stats, "Title", "inner") \
    .select("book_id", "Title", "metadata_avg_rating", "calculated_mean_rating")

task_b.show(5, truncate=False)

Mounted at /content/drive
--- Loading Data from Google Drive ---

--- Fill Rate Report ---
Column 'Title': 100.00%
Column 'description': 67.78%
Column 'authors': 85.21%
Column 'image': 75.48%
Column 'previewLink': 88.78%
Column 'publisher': 64.27%
Column 'publishedDate': 88.09%
Column 'infoLink': 88.78%
Column 'categories': 80.60%
Column 'ratingsCount': 23.42%

--- Cleaning Data ---

Data Cleanliness Score: 87.92% (Rows retained: 186755/212404)

--- Industry Req: Controversial Books (High Std Dev) ---
+----------------------------------------------------------------------------------+------------------+------------+
|Title                                                                             |volatility        |review_count|
+----------------------------------------------------------------------------------+------------------+------------+
|Growing Kids God's Way__Biblical Ethics for Parenting (19 Audio Cassette Tape Set)|1.969727027129269 |76          |
|What the Bible Says abou

In [ ]:
from pyspark.sql.functions import col, count, when, isnan, regexp_extract, stddev, avg, desc
import datetime

# ====================================================
# REQUIREMENT (b): Fill Rate Report
# Meaning: What % of data is NOT missing/garbage?
# ====================================================
print("--- Requirement (b): Fill Rate Report ---")

# 1. Get total count of raw records
total_source_records = books_df.count()
print(f"Total Source Records: {total_source_records}")

def generate_fill_rate_report(df):
    print(f"{'Column Name':<20} | {'Fill Rate %':<10}")
    print("-" * 35)

    for c in df.columns:
        # Check for Null, NaN, or Empty String ("")
        missing_count = df.filter(
            col(c).isNull() | isnan(c) | (col(c) == "")
        ).count()

        # Calculation: (Total - Missing) / Total * 100
        fill_rate = ((total_source_records - missing_count) / total_source_records) * 100

        print(f"{c:<20} | {fill_rate:.2f}%")

generate_fill_rate_report(books_df)


# ====================================================
# PRE-REQUISITE: Data Cleaning (Required for Step e)
# ====================================================
# We must clean the data first to calculate the cleanliness score.

# 1. Clean Year: Extract 4 digits from messy date strings
books_cleaning = books_df.withColumn("year_cleaned", regexp_extract(col("publishedDate"), r"(\d{4})", 1).cast("int"))

# 2. Define Cleaning Logic
#    - Rule 1: Title must not be Null (Critical Data)
#    - Rule 2: Year must be valid (Between 1800 and Current Year)
current_year = datetime.datetime.now().year

books_cleaned_df = books_cleaning.filter(
    (col("Title").isNotNull()) &
    (col("year_cleaned") > 1800) &
    (col("year_cleaned") <= current_year)
)

# Optimization: Cache the clean data for future steps
books_cleaned_df.cache()


# ====================================================
# REQUIREMENT (e): Data Cleanliness Percentage
# Formula: (Clean Records / Source Records) * 100
# ====================================================
print("\n--- Requirement (e): Data Cleanliness Score ---")

clean_records_count = books_cleaned_df.count()

cleanliness_percentage = (clean_records_count / total_source_records) * 100

print(f"Source Record Count : {total_source_records}")
print(f"Clean Record Count  : {clean_records_count}")
print(f"Data Cleanliness    : {cleanliness_percentage:.2f}%")
print(f"Rows Dropped        : {total_source_records - clean_records_count}")


# ====================================================
# REQUIREMENT (f): Industry Requirement
# Proposal: "Identify 'Controversial' Books (High Rating Volatility)"
# ====================================================
print("\n--- Requirement (f): Industry Requirement (Controversial Books) ---")
print("Objective: Identify books where users are highly divided (e.g., half vote 1-star, half vote 5-stars).")
print("Metric: Standard Deviation of Ratings > 1.5\n")

# 1. Group by Title and calculate Standard Deviation (Volatility) and Review Count
volatility_df = ratings_df.groupBy("Title").agg(
    stddev("review/score").alias("rating_volatility"),
    avg("review/score").alias("avg_score"),
    count("Title").alias("review_count")
)

# 2. Filter for statistical significance (ignore books with < 50 reviews)
#    Sort by highest volatility first
controversial_books = volatility_df.filter(col("review_count") > 50) \
                                   .orderBy(col("rating_volatility").desc())

print("Top 5 Most Controversial Books (High Volatility):")
controversial_books.show(5, truncate=False)

--- Requirement (b): Fill Rate Report ---
Total Source Records: 212404
Column Name          | Fill Rate %
-----------------------------------
Title                | 100.00%
description          | 67.78%
authors              | 85.21%
image                | 75.48%
previewLink          | 88.78%
publisher            | 64.27%
publishedDate        | 88.09%
infoLink             | 88.78%
categories           | 80.60%
ratingsCount         | 23.42%

--- Requirement (e): Data Cleanliness Score ---
Source Record Count : 212404
Clean Record Count  : 186755
Data Cleanliness    : 87.92%
Rows Dropped        : 25649

--- Requirement (f): Industry Requirement (Controversial Books) ---
Objective: Identify books where users are highly divided (e.g., half vote 1-star, half vote 5-stars).
Metric: Standard Deviation of Ratings > 1.5

Top 5 Most Controversial Books (High Volatility):
+----------------------------------------------------------------------------------+------------------+------------------+-----